In [ ]:
import torch
import torch.nn as nn
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import time
import random
import torch.nn.functional as F
import mujoco
import mujoco.viewer

In [ ]:
#functions to convert between real and simulated robot position or velocity
def sim2real(position):
    position[2] *= -1
    position[3], position[4] = position[4], position[3]
    position[6] *=2
    state = position[:-1]
    return state

def real2sim(state):
    position = state.copy()
    position[2] *= -1
    position[3], position[4] = position[4], position[3]
    position[6] /= 2
    position = np.append(position, position[6])
    return position

In [ ]:
class CobotEnv(gym.Env):
    def __init__(self, render_mode = None):
        xml_path= "/home/aaron-dsouza/programming/digital_twin/robots/mobile_aloha_sim/aloha_mujoco/aloha/meshes_mujoco/aloha_v1.xml"
        self.render_mode = render_mode
        self.model = mujoco.MjModel.from_xml_path(xml_path)
        self.data = mujoco.MjData(self.model)
        self.steps = 0
        self.max_torque = 2.0
        self.max_speed = 8.0
        self.max_steps = 2000
        self.viewer = None

        motor_qpos_low = np.array([-3.14158, 0, -3.14158, -2, -1.5708, -3.14158, 0])
        motor_qpos_high = np.array([3.14158, 3.14158, 0, 1.67, 1.5708, 3.14158, 0.95])

        motor_qvel_low = np.full(7, -8.0)
        motor_qvel_high = np.full(7, 8.0)
        
        self.cube_pos_low = np.array([0.45, -0.4, 0.7])
        self.cube_pos_high = np.array([0.95, 0.2, 1.5])
        
        self.disk_pos_low = np.array([-0.2, -0.65, 0.7])
        self.disk_pos_high = np.array([0.2, -0.05, 0.8])

        gripper_pos_low = np.array([-0.1, -0.5, 0.7])
        gripper_pos_high = np.array([1.2, 0.4, 2.0])

        all_obs_low = np.concatenate((motor_qpos_low, motor_qvel_low, self.cube_pos_low, self.disk_pos_low, gripper_pos_low))
        all_obs_high = np.concatenate((motor_qpos_high, motor_qvel_high, self.cube_pos_high, self.disk_pos_high, gripper_pos_high))

        self.action_space = spaces.Box(
            low=motor_qpos_low, high=motor_qpos_high, dtype=np.float32
        )
        self.observation_space = spaces.Box(
            low=all_obs_low, high=all_obs_high, dtype=np.float32
        )

    def get_observation(self):
        motor_qpos = sim2real(self.data.qpos[-8:])
        motor_qvel = sim2real(self.data.qvel[-8:])
        cube_pos = self.data.qpos[:3]
        disk_pos = self.data.qpos[7:10]
        gripper_pos = self.data.site_xpos[1]
        all_observations = np.concatenate((motor_qpos, motor_qvel, cube_pos, disk_pos, gripper_pos), dtype=np.float32)
        return all_observations

    def apply_action(self, action):
        action = np.clip(action, self.action_space.low, self.action_space.high)
        self.data.ctrl[-8:] = real2sim(action)
        mujoco.mj_step(self.model, self.data)

    def compute_rewards(self):
        raise NotImplementedError
        # theta = (self.data.qpos[0] + np.pi) % (2 * np.pi) - np.pi   # normalize to [-pi, pi]
        # reward = -(theta**2 + 0.1 * self.data.qvel[0]**2 + 0.001 * self.data.ctrl[0]**2)
        # return reward

    def reset(self):
        mujoco.mj_resetData(self.model, self.data)

        cube_x = np.random.uniform(self.cube_pos_low[0], self.cube_pos_high[0])
        cube_y = np.random.uniform(self.cube_pos_low[1], self.cube_pos_high[1])
        self.data.qpos[:2] = np.array([cube_x, cube_y])

        disk_x = np.random.uniform(self.disk_pos_low[0], self.disk_pos_high[0])
        disk_y = np.random.uniform(self.disk_pos_low[1], self.disk_pos_high[1])
        self.data.qpos[7:9] = np.array([disk_x, disk_y])

        mujoco.mj_forward(self.model, self.data)
        self.steps = 0
        return self.get_observation(), {}
        
    def step(self, action):
        self.apply_action(action)
        next_state = self.get_observation()
        reward = self.compute_rewards()
        self.steps +=1
        terminated = np.max(np.abs(self.data.qacc)) > 1e6
        truncated = self.steps >= self.max_steps

        if self.render_mode == 'human':
            self.render()
        
        #next_state, reward, terminated, truncated, info =
        return next_state, reward, terminated, truncated, {}

    def render(self):
        if self.viewer == None:
            self.viewer = mujoco.viewer.launch_passive(self.model, self.data)

        self.viewer.sync()

        if not self.viewer.is_running():
            self.viewer.close()
            self.viewer = None

    def close(self):
        if self.viewer is not None:
            self.viewer.close()
            self.viewer = None

In [ ]:
class CobotApproachEnv(CobotEnv):
    def __init__(self, render_mode = None):
        super().__init__(render_mode)

    def compute_rewards(self):
        gripper_pos = self.data.site_xpos[1]
        cube_pos = self.data.qpos[:3]
        distance = np.sum((gripper_pos-cube_pos)**2)
        velocity_cost = 0.001 * np.sum(self.data.qvel[-8:]**2)
        reward = -(distance + velocity_cost)
        return reward

In [ ]:
# env = CobotApproachEnv(render_mode='human')
# env.render()
# episodeNumber = 10
# timesteps = 100
# for episodeIndex in range(episodeNumber):
#     initial_state= env.reset()
#     for timeIndex in range(timesteps):
#         rand_action = env.action_space.sample()
#         observation, reward, terminated, truncated, info = env.step(rand_action)
#         print(reward)
#         time.sleep(0.1)
#         if(terminated or truncated):
#             time.sleep(1)
#             break

# env.close()

/home/aaron-dsouza/programming/miniconda3/envs/mujoco/lib/python3.12/site-packages/gymnasium/spaces/box.py:231: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/home/aaron-dsouza/programming/miniconda3/envs/mujoco/lib/python3.12/site-packages/gymnasium/spaces/box.py:297: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/home/aaron-dsouza/programming/miniconda3/envs/mujoco/lib/python3.12/site-packages/glfw/__init__.py:917: GLFWError: (65548) b'Wayland: The platform does not provide the window position'
  warnings.warn(message, GLFWError)


-57.48208552744546
-26.961053679629075
-8.766535772102202
-4.248603444323184
-2.6748639475862284
-2.4302112590035523
-1.1107296542164486
-37.94431602267457
-38.09148278697458
-8.39202006824644
-8.681989764481129
-7.059640546841193
-10.40192283695567
-5.2677676505952675
-4.611900038732197
-11.323120450812752
-20.667097550066487
-86.7101160400795
-416.14003576635974
-2095.2294691817956
-9577.071406830673
-41.66531693312711
-16.427824937205607
-3.9939047087977553
-37.30466319970985
-31.205708079759212
-34.86432039999311
-15.59665907285222
-19.00836866593816
-11.820963193073627
-8.017873892061386
-6.202750569830427
-5.137556996967454
-6.33481009474766
-2.5797120335313704
-4.521055016513362
-7.011736598317707
-7.2211545387662
-6.590244368056397
-5.827077510218025
-29.17613799160099
-294.3524310917917
-2227.2308323800185
-123170.49631210494


In [ ]:
from collections import deque

class ReplayBuffer:
    def __init__(self, buffer_size=50000):
        self.buffer = deque(maxlen=buffer_size)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = map(np.stack, zip(*batch))
        return states, actions, rewards, next_states, dones

    def __len__(self):
        return len(self.buffer)

In [ ]:
class CriticNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dims):
        super().__init__()
        self.critic1= nn.Sequential(
            nn.Linear(state_dim+action_dim, hidden_dims[0]),
            nn.ReLU(),
            nn.Linear(hidden_dims[0], hidden_dims[1]),
            nn.ReLU(),
            nn.Linear(hidden_dims[1], 1)
        )
        self.critic2 = nn.Sequential(
            nn.Linear(state_dim+action_dim, hidden_dims[0]),
            nn.ReLU(),
            nn.Linear(hidden_dims[0], hidden_dims[1]),
            nn.ReLU(),
            nn.Linear(hidden_dims[1], 1),
        )

    def forward(self, state, action):
        inp = torch.cat([state, action], dim=-1)
        Q1 = self.critic1(inp)
        Q2 = self.critic2(inp)
        return Q1, Q2

In [ ]:
class ActorNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dims, action_low, action_high, initial_alpha=0.2):
        super().__init__()
        self.action_scale = torch.as_tensor((action_high - action_low)/2, dtype = torch.float32)
        self.action_bias = torch.as_tensor((action_high + action_low)/2, dtype = torch.float32)
        self.feature_head = nn.Sequential(
                    nn.Linear(state_dim, hidden_dims[0]),
                    nn.ReLU(),
                    nn.Linear(hidden_dims[0], hidden_dims[1]),
                    nn.ReLU(),
                )
        self.mean_head = nn.Linear(hidden_dims[1], action_dim)
        self.log_std_head = nn.Linear(hidden_dims[1], action_dim)

    def forward(self, state):
        features = self.feature_head(state)
        mean = self.mean_head(features)
        log_std = self.log_std_head(features)

        log_std = torch.clamp(log_std, -20, 2)
        std = torch.exp(log_std)
        dist = torch.distributions.Normal(mean, std)
        sample = dist.rsample()
        action = torch.tanh(sample) 

        log_prob = dist.log_prob(sample)
        log_prob -= torch.log(1-action.pow(2) + 1e-6)
        log_prob = log_prob.sum(dim=-1, keepdim=True)
        action = action * self.action_scale + self.action_bias
    
        return action, log_prob

    def sample(self, state):
        features = self.feature_head(state)
        mean = self.mean_head(features)
        action = torch.tanh(mean)
        action = action * self.action_scale + self.action_bias

In [ ]:
class SoftActorCritic:
    def __init__(self):
        self.env = CobotApproachEnv()

        state_dim = 7+7+3+3+3
        action_dim = 7
        hidden_dim = (256, 256)

        self.critic = CriticNetwork(state_dim, action_dim, hidden_dim)
        self.actor = ActorNetwork(state_dim, action_dim, hidden_dim, 
                                  self.env.action_space.low, self.env.action_space.high)
        initial_alpha = 0.2
        
        self.log_alpha = torch.tensor(
            np.log(initial_alpha),
            dtype=torch.float32,
            requires_grad=True
        )
        self.target_entropy = -action_dim
        self.target_critic = CriticNetwork(state_dim, action_dim, hidden_dim)
        self.target_critic.load_state_dict(self.critic.state_dict())
        for param in self.target_critic.parameters():
            param.requires_grad = False

        self.replay_buffer = ReplayBuffer(200_000)

        self.gamma = 0.98
        self.batch_size = 256

        self.critic_optimizer = torch.optim.Adam(self.critic.parameters(), lr=4e-4)
        self.actor_optimizer = torch.optim.Adam(self.actor.parameters(), lr=4e-4)
        self.alpha_optimizer = torch.optim.Adam([self.log_alpha], lr=3e-4)

    @torch.no_grad()
    def get_target(self, next_states, rewards, dones):
        next_states = torch.as_tensor(next_states, dtype=torch.float32)
        rewards = torch.as_tensor(rewards, dtype=torch.float32).unsqueeze(1)
        dones = torch.as_tensor(dones, dtype=torch.float32).unsqueeze(1)
        next_actions, log_prob = self.actor(next_states)

        target_Q1, target_Q2 = self.target_critic(next_states, next_actions)
        target_Q = torch.min(target_Q1, target_Q2)
        target = rewards + self.gamma * (1-dones) * (
            target_Q - self.alpha * log_prob
        )
        return target

    def update_critic(self, targets, states, actions):
        states = torch.as_tensor(states, dtype=torch.float32)
        actions = torch.as_tensor(actions, dtype=torch.float32)

        Q1, Q2 = self.critic(states, actions)

        critic_loss = F.mse_loss(Q1, targets) + F.mse_loss(Q2, targets)   

        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        self.critic_optimizer.step()

    def update_actor(self, states):
        states = torch.as_tensor(states, dtype=torch.float32)

        new_actions, log_prob = self.actor(states)
        Q1, Q2 = self.critic(states, new_actions)
        Q = torch.min(Q1, Q2)

        actor_loss = (self.log_alpha.exp() * log_prob - Q).mean()

        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        self.actor_optimizer.step()

        alpha_loss = -(self.log_alpha * (log_prob + self.target_entropy).detach()).mean()
        self.alpha_optimizer.zero_grad()
        alpha_loss.backward()
        self.alpha_optimizer.step()


    def soft_target_update(self, tau=0.005):
        for target_param, critic_param in zip(self.target_critic.parameters(), 
                                            self.critic.parameters()):
            target_param.data.copy_(
                tau * critic_param.data +
                (1-tau) * target_param.data
            )

    def train(self, n_iterations):
        self.actor.train()
        self.critic.train()
        state, _ = self.env.reset()
        state = torch.as_tensor(state, dtype=torch.float32).unsqueeze(0)
        episode_reward = 0
        for step in range(n_iterations):
            if step < 10000:
                action = self.env.action_space.sample()
            else:
                action, _ = self.actor(state)
                action = action.squeeze(0).detach().numpy()
            next_state, reward, terminated, truncated, _ = self.env.step(action)
            episode_reward += reward
            done = terminated or truncated
            self.replay_buffer.push(state.squeeze(0).numpy(), action, reward, next_state, done)

            if len(self.replay_buffer) >= self.batch_size:
                states, actions, rewards, next_states, dones = self.replay_buffer.sample(self.batch_size)
                
                targets = self.get_target(next_states, rewards, dones)
                self.update_critic(targets, states, actions)
                self.update_actor(states)
                self.soft_target_update()

            if done:    
                print(f"Step {step}: {episode_reward:.2f}")
                episode_reward = 0
                state, _ = self.env.reset()
                state = torch.as_tensor(state, dtype=torch.float32).unsqueeze(0)
                
            else:
                state = torch.as_tensor(next_state, dtype=torch.float32).unsqueeze(0)

    @torch.no_grad()
    def eval(self):
        self.actor.eval()
        eval_env = CobotApproachEnv(render_mode="human")
        state, _ = eval_env.reset()
        state = torch.as_tensor(state, dtype=torch.float32).unsqueeze(0)
        done = False
        while(not done):
            action = self.actor.sample(state)
            action = action.squeeze(0).detach().numpy()
            next_state, reward, terminated, truncated, _ = eval_env.step(action)
            done = terminated or truncated
            state = torch.as_tensor(next_state, dtype=torch.float32).unsqueeze(0)
            time.sleep(0.01)
        eval_env.close()

In [ ]:
sac = SoftActorCritic()
# state_dict = torch.load("sac_checkpoint.pt", weights_only=True)
# sac.actor.load_state_dict(state_dict["actor"])
sac.train(100)
sac.eval()

In [ ]:
# for i in range(10):
#     sac.eval()

: 